# 🔧 Notebook 03 — Feature Engineering
## Bagian 3: Ekstraksi Fitur (Feature Extraction)

**Metode yang diimplementasikan:**
- Statistik: TF-IDF, BM25
- Word Embeddings: Word2Vec, GloVe, FastText
- Transformer: DistilBERT, IndoBERT, RoBERTa

**Subset Prioritas:** TF-IDF, FastText, IndoBERT

## Setup

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import scipy.sparse as sp
import time
from sklearn.model_selection import train_test_split

from src.feature_extractors import (
    TFIDFExtractor, BM25Extractor, Word2VecExtractor,
    GloVeExtractor, FastTextExtractor, DistilBERTExtractor,
    BERTExtractor, RoBERTaExtractor, get_all_extractors
)

## Load Data

In [2]:
df = pd.read_csv('../data/processed/reviews_prepared.csv')
print(f"📊 Dataset: {df.shape[0]} baris")

# Pastikan kolom yang dibutuhkan ada
required_cols = ['review_clean', 'sentiment_encoded', 'sentiment']
for col in required_cols:
    if col not in df.columns:
        print(f"❌ Kolom '{col}' tidak ditemukan! Jalankan Notebook 02 terlebih dahulu.")

📊 Dataset: 609 baris


In [3]:
# Prepare data
texts = df['review_clean'].fillna('').tolist()
labels = df['sentiment_encoded'].values

# Train-test split (stratified)
texts_train, texts_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"\n📊 Split: Train={len(texts_train)}, Test={len(texts_test)}")
print(f"   Train labels: {np.bincount(y_train)} (Neg/Net/Pos)")
print(f"   Test labels:  {np.bincount(y_test)} (Neg/Net/Pos)")


📊 Split: Train=487, Test=122
   Train labels: [163 162 162] (Neg/Net/Pos)
   Test labels:  [40 41 41] (Neg/Net/Pos)


## 1. TF-IDF (3.1)

In [4]:
print("\n" + "="*60)
print("📦 1. TF-IDF Feature Extraction")
print("="*60)

tfidf = TFIDFExtractor(max_features=10000, ngram_range=(1, 2))
X_train_tfidf, X_test_tfidf = tfidf.fit_transform(texts_train, texts_test)

# Top terms
print("\n📊 Top 20 terms (highest TF-IDF):")
top_terms = tfidf.get_top_terms(20)
for term, score in top_terms:
    print(f"   {term:20s} → IDF={score:.4f}")

# Save
os.makedirs('../results/feature_matrices', exist_ok=True)
sp.save_npz('../results/feature_matrices/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('../results/feature_matrices/X_test_tfidf.npz', X_test_tfidf)
print(f"\n💾 Saved TF-IDF features")


📦 1. TF-IDF Feature Extraction
   TF-IDF: vocab=795, shape=(487, 795)

📊 Top 20 terms (highest TF-IDF):
   adanya aplikasi      → IDF=6.0917
   ydah                 → IDF=6.0917
   wkwkwk               → IDF=6.0917
   when                 → IDF=6.0917
   websitenya           → IDF=6.0917
   agar tidak           → IDF=6.0917
   web tapi             → IDF=6.0917
   web kaya             → IDF=6.0917
   web aplikasi         → IDF=6.0917
   aktif                → IDF=6.0917
   akunnya              → IDF=6.0917
   wajib                → IDF=6.0917
   alias                → IDF=6.0917
   badan                → IDF=6.0917
   baca ulasan          → IDF=6.0917
   baca                 → IDF=6.0917
   ayolah               → IDF=6.0917
   bantuan              → IDF=6.0917
   baik aplikasi        → IDF=6.0917
   bagus lagi           → IDF=6.0917

💾 Saved TF-IDF features


## 2. FastText (3.3)

In [5]:
print("\n" + "="*60)
print("📦 2. FastText Feature Extraction (Train from Scratch)")
print("="*60)

fasttext_ext = FastTextExtractor(vector_size=100, train_from_scratch=True)
X_train_ft, X_test_ft = fasttext_ext.fit_transform(texts_train, texts_test)

# Save
np.save('../results/feature_matrices/X_train_fasttext.npy', X_train_ft)
np.save('../results/feature_matrices/X_test_fasttext.npy', X_test_ft)
print(f"\n💾 Saved FastText features")


📦 2. FastText Feature Extraction (Train from Scratch)
   FastText: trained from scratch, vocab=544
   FastText: shape=(487, 100)

💾 Saved FastText features


## 3. IndoBERT (3.7)

⚠️ **Catatan:** Tahap ini membutuhkan download model (~400MB) dan bisa
memakan waktu lama tanpa GPU. Jika timeout, skip dan gunakan TF-IDF/FastText saja.

In [6]:
# print("\n" + "="*60)
# print("📦 3. IndoBERT Feature Extraction (CLS token)")
# print("="*60)

# try:
#     bert_ext = BERTExtractor(
#         model_name='indobenchmark/indobert-base-p1',
#         max_length=128,
#         batch_size=32,
#         pooling='cls'
#     )
#     X_train_bert, X_test_bert = bert_ext.fit_transform(texts_train, texts_test)

#     np.save('../results/feature_matrices/X_train_indobert.npy', X_train_bert)
#     np.save('../results/feature_matrices/X_test_indobert.npy', X_test_bert)
#     print(f"\n💾 Saved IndoBERT features")

# except Exception as e:
#     print(f"\n⚠️ IndoBERT extraction gagal: {e}")
#     print("   Lanjutkan dengan TF-IDF dan FastText saja.")
#     X_train_bert, X_test_bert = None, None

## (Opsional) Feature Extractors Tambahan

Uncomment cell di bawah untuk menjalankan extractor tambahan.

In [7]:
# === WORD2VEC ===
print("\n" + "="*60)
print("📦 Word2Vec Feature Extraction")
print("="*60)
w2v_ext = Word2VecExtractor(vector_size=100, train_from_scratch=True)
X_train_w2v, X_test_w2v = w2v_ext.fit_transform(texts_train, texts_test)
np.save('../results/feature_matrices/X_train_word2vec.npy', X_train_w2v)
np.save('../results/feature_matrices/X_test_word2vec.npy', X_test_w2v)


📦 Word2Vec Feature Extraction
   Word2Vec: trained from scratch, vocab=544
   Word2Vec: OOV rate = 828/1372 (60.3%)
   Word2Vec: shape=(487, 100)


In [8]:
# === BM25 ===
# print("\n" + "="*60)
# print("📦 BM25 Feature Extraction")
# print("="*60)
# bm25_ext = BM25Extractor()
# X_train_bm25, X_test_bm25 = bm25_ext.fit_transform(texts_train, texts_test, train_labels=y_train)
# np.save('../results/feature_matrices/X_train_bm25.npy', X_train_bm25)
# np.save('../results/feature_matrices/X_test_bm25.npy', X_test_bm25)

In [9]:
# === GloVe (cc.id.300.vec) ===
print("\n" + "="*60)
print("📦 GloVe Feature Extraction")
print("="*60)
glove_ext = GloVeExtractor(vectors_path='../data/embeddings/cc.id.300.vec', dim=300)
X_train_glove, X_test_glove = glove_ext.fit_transform(texts_train, texts_test)
np.save('../results/feature_matrices/X_train_glove.npy', X_train_glove)
np.save('../results/feature_matrices/X_test_glove.npy', X_test_glove)


📦 GloVe Feature Extraction
   GloVe: Loading vectors dari ../data/embeddings/cc.id.300.vec...


   GloVe: vocab loaded=200000
   GloVe: OOV rate = 183/1372 (13.3%)
   GloVe: shape=(487, 300)


## Ringkasan Feature Extraction

In [10]:
print("\n📊 Ringkasan Feature Matrices:")
print("=" * 60)
features_summary = {
    'TF-IDF': X_train_tfidf.shape if X_train_tfidf is not None else 'N/A',
    'FastText': X_train_ft.shape if X_train_ft is not None else 'N/A',
    'Word2Vec': X_train_w2v.shape if 'X_train_w2v' in locals() else 'N/A',
    'Glove': X_train_glove.shape if 'X_train_glove' in locals() else 'N/A',
}
for name, shape in features_summary.items():
    ftype = 'sparse' if name == 'TF-IDF' else 'dense'
    print(f"   {name:15s} : {str(shape):20s} ({ftype})")

# Simpan y_train, y_test untuk notebook selanjutnya
np.save('../results/feature_matrices/y_train.npy', y_train)
np.save('../results/feature_matrices/y_test.npy', y_test)
print(f"\n💾 Labels saved: y_train ({len(y_train)}), y_test ({len(y_test)})")


📊 Ringkasan Feature Matrices:
   TF-IDF          : (487, 795)           (sparse)
   FastText        : (487, 100)           (dense)
   Word2Vec        : (487, 100)           (dense)
   Glove           : (487, 300)           (dense)

💾 Labels saved: y_train (487), y_test (122)


In [11]:
print("\n✅ Feature extraction selesai! Siap untuk Model Training (Notebook 04).")


✅ Feature extraction selesai! Siap untuk Model Training (Notebook 04).
